**Tumour-Only WT Matching for SWI/SNF Loss Cases**
This notebook creates the matched case-control table for TE differential expression.

Comparison:

SWI/SNF-loss primary tumour vs SWI/SNF-WT primary tumour
Controls are matched by: cancer type / indication, sex

WT controls are restricted to TCGA sample type code 01, meaning primary tumour.

In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(Biobase)
library(stringr)
library(purrr)

setwd("/home/kennes38/ResearchProject/03_swi_snf_WTmatching")

gene_set_label <- "swi_snf_loss"

tedata_match_input_dir <- "../03_swi_snf_TEdata_matching"

tedata_root <- "../TEdata_resources"

case_file <- file.path(
  tedata_match_input_dir,
  paste0("pancancer_", gene_set_label, "_samples_with_RNA_match_TE_ready.csv")
)

match_output_dir <- "."

n_controls_per_case <- 2

set.seed(1)

file.exists(case_file)
dir.exists(tedata_root)
dir.exists(match_output_dir)

In [ ]:
# Load REdiscoverTE data and metadata

rds_hits <- list.files(
  tedata_root,
  pattern = "eset_TCGA_TE_intergenic_logCPM\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

stopifnot(length(rds_hits) == 1)
dat <- readRDS(rds_hits[1])

pdat <- pData(dat) %>%
  tibble::rownames_to_column("TEdata_full_id")

tedata_annot <- tibble(
  # Exact REdiscoverTE sample identifier
  TEdata_full_id = sampleNames(dat),

  # TCGA sample/patient identifiers derived from the sample barcode
  sample_id_16 = substr(sampleNames(dat), 1, 16),
  patient_id = substr(sampleNames(dat), 1, 12),

  # 01 = primary tumour
  sample_type_code = substr(sample_id_16, 14, 15)
) %>%
  left_join(pdat, by = "TEdata_full_id")

colnames(tedata_annot)

In [ ]:
# Define matching variables

# REdiscoverTE metadata columns used in your previous workflow
project_col <- "indication"
sex_col <- "Sex"

stopifnot(project_col %in% colnames(tedata_annot))
stopifnot(sex_col %in% colnames(tedata_annot))

tedata_match <- tedata_annot %>%
  # match_project = tumour type / cancer indication
  # match_sex = sample sex
  mutate(
    match_project = as.character(.data[[project_col]]),
    match_sex = as.character(.data[[sex_col]]),
    match_project = na_if(match_project, ""),
    match_sex = na_if(match_sex, "")
  )

tedata_match %>% count(sample_type_code, sort = TRUE)
tedata_match %>% count(match_project, sort = TRUE)
tedata_match %>% count(match_sex, sort = TRUE)

In [ ]:
# Load loss cases and mark case samples

loss_cases <- read_csv(case_file, show_col_types = FALSE) %>%
  # Add TCGA sample type code so cases can also be restricted to primary tumours
  mutate(
    sample_id_16 = substr(as.character(sample_id_16), 1, 16),
    patient_id = substr(as.character(patient_id), 1, 12),
    case_sample_type_code = substr(sample_id_16, 14, 15)
  ) %>%
  distinct()

cat("Loss cases with TEdata match:", nrow(loss_cases), "\n")

# Keep primary tumour cases only for the main analysis
loss_cases_primary <- loss_cases %>%
  # Main analysis keeps only primary tumour altered cases
  filter(case_sample_type_code == "01")

cat("Primary tumour loss cases:", nrow(loss_cases_primary), "\n")

loss_cases_primary %>%
  count(project, sort = TRUE)

In [ ]:
# Create case metadata in TEdata

case_metadata <- loss_cases_primary %>%
  left_join(
    tedata_match %>%
      select(
        sample_id_16,
        patient_id,
        match_project,
        match_sex,
        TEdata_sample_type_code = sample_type_code
      ) %>%
      distinct(),
    by = c("sample_id_16", "patient_id")
  ) %>%
  filter(!is.na(TEdata_full_id)) %>%
  distinct(sample_id_16, .keep_all = TRUE)

cat("Primary tumour loss cases found in TEdata:", nrow(case_metadata), "\n")

case_metadata %>%
  count(match_project, sort = TRUE)

In [ ]:
# Create primary tumour WT-pool

case_sample_ids <- case_metadata$sample_id_16
case_patient_ids <- case_metadata$patient_id

wt_pool <- tedata_match %>%
  # Controls must be primary tumours
  filter(sample_type_code == "01") %>%
  # Do not allow an altered case sample to be selected as a WT control
  filter(!sample_id_16 %in% case_sample_ids) %>%
  # Do not use another sample from the same patient as a control
  filter(!patient_id %in% case_patient_ids) %>%
  select(
    TEdata_full_id,
    sample_id_16,
    patient_id,
    WT_sample_type_code = sample_type_code,
    match_project,
    match_sex
  ) %>%
  distinct()

cat("Primary tumour WT candidates:", nrow(wt_pool), "\n")

wt_pool %>%
  count(match_project, sort = TRUE)

In [ ]:
# Match controls

match_one_case <- function(case_row, wt_df, n_controls = 2) {
  # First match on tumour type
  candidates <- wt_df %>%
    filter(match_project == case_row$match_project)

  # Then match on sex, if sex is available for the case
  if (!is.na(case_row$match_sex) && case_row$match_sex != "") {
    candidates <- candidates %>%
      filter(match_sex == case_row$match_sex)
  }

  if (nrow(candidates) == 0) {
    return(tibble())
  }

  candidates %>%
    # Randomly sample controls -> same tumour type does not always contribute the first rows in the ExpressionSet
    slice_sample(n = min(n_controls, nrow(candidates))) %>%
    mutate(
      case_TEdata_full_id = case_row$TEdata_full_id,
      case_sample_id_16 = case_row$sample_id_16,
      case_patient_id = case_row$patient_id,
      case_sample_type_code = case_row$case_sample_type_code,
      case_project = case_row$match_project,
      case_sex = case_row$match_sex,
      genes_lost = case_row$genes_lost,
      events = case_row$events,
      lof_genes = case_row$lof_genes,
      homdel_genes = case_row$homdel_genes,
      mutation_classes = case_row$mutation_classes,
      n_genes_lost = case_row$n_genes_lost,
      n_loss_events = case_row$n_loss_events,
      has_lof = case_row$has_lof,
      has_homdel = case_row$has_homdel
    )
}

case_rows <- split(case_metadata, seq_len(nrow(case_metadata)))

matched_wt <- purrr::map_dfr(
  case_rows,
  ~ match_one_case(.x, wt_pool, n_controls = n_controls_per_case)
)

cat("Matched WT rows:", nrow(matched_wt), "\n")

In [ ]:
# Summarise matching success
match_counts <- matched_wt %>%
  # Count how many controls were found for each altered case
  count(case_sample_id_16, name = "n_controls_found")

case_match_summary <- case_metadata %>%
  left_join(match_counts, by = c("sample_id_16" = "case_sample_id_16")) %>%
  mutate(
    n_controls_found = ifelse(is.na(n_controls_found), 0L, n_controls_found),
    matched_successfully = n_controls_found > 0
  )

case_match_summary %>%
  count(matched_successfully)

case_match_summary %>%
  count(match_project, matched_successfully, sort = TRUE)

In [ ]:
# Create final case-control table

case_control_tbl <- matched_wt %>%
  # Final table has one row per case-control pair
  # Case with two controls appears in two rows
  select(
    case_TEdata_full_id,
    case_sample_id_16,
    case_patient_id,
    case_sample_type_code,
    case_project,
    case_sex,
    genes_lost,
    events,
    lof_genes,
    homdel_genes,
    mutation_classes,
    n_genes_lost,
    n_loss_events,
    has_lof,
    has_homdel,
    WT_TEdata_full_id = TEdata_full_id,
    WT_sample_id_16 = sample_id_16,
    WT_patient_id = patient_id,
    WT_sample_type_code,
    WT_project = match_project,
    WT_sex = match_sex
  )

# Confirm controls are primary tumour only
case_control_tbl %>%
  count(case_sample_type_code, WT_sample_type_code)

case_control_tbl %>%
  summarise(
    n_cases = n_distinct(case_sample_id_16),
    n_controls = n_distinct(WT_sample_id_16),
    n_rows = n()
  )

In [ ]:
# Save matching outputs

write_csv(
  case_match_summary,
  file.path(match_output_dir, paste0(gene_set_label, "_case_match_summary_cancer_sex_primary_tumour_only.csv"))
)

write_csv(
  matched_wt,
  file.path(match_output_dir, paste0(gene_set_label, "_matched_WT_controls_cancer_sex_primary_tumour_only_raw.csv"))
)

write_csv(
  case_control_tbl,
  file.path(match_output_dir, paste0(gene_set_label, "_case_control_matched_table_cancer_sex_primary_tumour_only.csv"))
)